# Walmart Recruiting - Store Sales Forecasting


<img src='https://ares.shiftdelete.net/2024/10/walmart.png'>


Bu çalışmada `Walmart Recruiting - Store Sales Forecasting` yarışması için mağaza, tarih ve dışsal özellikler kullanılarak satış tahmini yapan bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Veri birleştirme ve ön işleme
5. Özellik çıkarımı
6. Model kurma
7. RMSE değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
!pip install catboost -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import io
import os
import zipfile

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [2]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [3]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/walmart-recruiting-store-sales-forecasting.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

def find_zip_member(members, candidates):
    for candidate in candidates:
        for member in members:
            if member.endswith(candidate):
                return member
    raise FileNotFoundError(f'{candidates[0]} zip içinde bulunamadi.')

train_member = find_zip_member(zip_members, ['train.csv', 'train.csv.zip'])
test_member = find_zip_member(zip_members, ['test.csv', 'test.csv.zip'])
features_member = find_zip_member(zip_members, ['features.csv', 'features.csv.zip'])
stores_member = find_zip_member(zip_members, ['stores.csv', 'stores.csv.zip'])
sample_submission_member = find_zip_member(zip_members, ['sampleSubmission.csv', 'sampleSubmission.csv.zip'])


Mounted at /content/drive


## 3. Veri Dosyalarını Yükleme


In [3]:
# Bu bölümde yarışma dosyalarını doğrudan zip içinden yüklüyoruz.


In [4]:
def read_csv_from_zip(zip_path, member_name):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            if member_name.endswith('.zip'):
                inner_bytes = f.read()
                with zipfile.ZipFile(io.BytesIO(inner_bytes), 'r') as inner_zip:
                    inner_name = inner_zip.namelist()[0]
                    with inner_zip.open(inner_name) as inner_f:
                        return pd.read_csv(inner_f)
            return pd.read_csv(f)

train = read_csv_from_zip(zip_path, train_member)
test = read_csv_from_zip(zip_path, test_member)
features = read_csv_from_zip(zip_path, features_member)
stores = read_csv_from_zip(zip_path, stores_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Features shape:', features.shape)
print('Stores shape:', stores.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape: (421570, 5)
Test shape: (115064, 4)
Features shape: (8190, 12)
Stores shape: (45, 3)
Sample submission shape: (115064, 2)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


## 4. Veri Birleştirme ve Ön İşleme


In [4]:
# Bu bölümde tarih, mağaza ve dışsal özellikleri birleştiriyoruz.


In [5]:
train_df = train.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
train_df = train_df.merge(stores, on='Store', how='left')

test_df = test.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
test_df = test_df.merge(stores, on='Store', how='left')

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (421570, 16)
Test merged shape: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


## 5. Özellik Çıkarımı


In [5]:
# Bu bölümde tarih bilgisinden ek özellikler çıkarıyoruz.


In [6]:
for df in [train_df, test_df]:
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['day'] = df['Date'].dt.day

feature_cols = [
    'Store', 'Dept', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2',
    'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Type', 'Size',
    'year', 'month', 'week', 'day'
]

target_col = 'Weekly_Sales'

train_df[target_col] = pd.to_numeric(train_df[target_col], errors='coerce')
train_df = train_df[train_df[target_col].notna()].copy()
train_df = train_df[train_df[target_col] > -1].copy()

train_x = train_df[feature_cols].copy()
train_y = np.log1p(train_df[target_col])
test_x = test_df[feature_cols].copy()

categorical_cols = ['Type']
for col in categorical_cols:
    train_x[col] = train_x[col].astype(str)
    test_x[col] = test_x[col].astype(str)

numeric_cols = [col for col in feature_cols if col not in categorical_cols]
for col in numeric_cols:
    train_x[col] = pd.to_numeric(train_x[col], errors='coerce')
    test_x[col] = pd.to_numeric(test_x[col], errors='coerce')

x_train, x_valid, y_train, y_valid = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

print('x_train shape:', x_train.shape)
print('x_valid shape:', x_valid.shape)


x_train shape: (336267, 18)
x_valid shape: (84067, 18)


## 6. Model Kurma


In [6]:
# Bu bölümde haftalık satış tahmini için CatBoost modelini kuruyoruz.


In [7]:
model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.08,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=0,
    random_seed=42
)

model.fit(
    x_train,
    y_train,
    cat_features=categorical_cols,
    eval_set=(x_valid, y_valid),
    use_best_model=True
)


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=300, learning_rate=0.08, loss_function='RMSE', random_seed=42, verbose=0)

## 7. RMSE Değerlendirmesi


In [7]:
# Bu bölümde validation verisi üzerindeki RMSE değerini hesaplıyoruz.


In [8]:
valid_preds = model.predict(x_valid)
rmse = root_mean_squared_error(np.expm1(y_valid), np.expm1(valid_preds))
print('Validation RMSE:', round(rmse, 5))


Validation RMSE: 9446.2639


## 8. Test Tahmini ve Submission


In [8]:
# Bu bölümde test tahminlerini submission formatına dönüştürüyoruz.


In [9]:
test_preds = model.predict(test_x)
test_sales = np.expm1(test_preds)

submission = sample_submission.copy()
if 'Weekly_Sales' in submission.columns:
    submission['Weekly_Sales'] = test_sales[:len(submission)]
else:
    submission.iloc[:, 1] = test_sales[:len(submission)]

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (115064, 2)


,Id,Weekly_Sales
0,1_1_2012-11-02,22184.356469
1,1_1_2012-11-09,21886.702951
2,1_1_2012-11-16,22937.776902
3,1_1_2012-11-23,29425.113241
4,1_1_2012-11-30,26412.016914


In [10]:
output_path = '/content/walmart_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/walmart_submission.csv


## 9. Sonuç


Bu çalışmada Walmart Recruiting - Store Sales Forecasting yarışması için mağaza, tarih ve dışsal özellikler kullanılarak haftalık satış tahmini yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 9446.26390 RMSE değeri elde etti ve test verisi için haftalık satış tahminleri üretildi.


In [ ]:
# Bu bölümde elde edilen tahmin sonucunu kısaca değerlendiriyoruz.
